# Multi-Stage Progress Example with SSE

> Demonstrates a multi-stage data processing pipeline with hierarchical progress tracking using Server-Sent Events (SSE), featuring real-time stage indicators, nested tqdm progress bars with configurable throttling, and HTMX SSE-powered UI state management for concurrent job control.

In [1]:
from fasthtml.common import *
import uuid, time, json
import random
import asyncio

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner
from cjm_tqdm_capture.streaming import sse_stream_async

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_behaviors
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors
from cjm_fasthtml_daisyui.components.data_display.card import card
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_sizes, badge_styles, badge_colors
from cjm_fasthtml_daisyui.components.data_input.range_slider import range_dui, range_colors
from cjm_fasthtml_daisyui.components.data_input.label import label
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.layout import display_tw
from cjm_fasthtml_tailwind.utilities.spacing import p, m, space
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size, text_align
from cjm_fasthtml_tailwind.utilities.sizing import w, h, container
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, flex_wrap, gap
from cjm_fasthtml_tailwind.utilities.borders import rounded
from cjm_fasthtml_tailwind.core.base import combine_classes

In [3]:
# Create app
app, rt = create_test_app(theme=DaisyUITheme.CYBERPUNK)
app.hdrs.append(Link(rel='icon', type='image/svg+xml', href=f'https://api.dicebear.com/9.x/adventurer/svg?seed={random.random()}'))

# Initialize
monitor = ProgressMonitor(keep_history=True)
runner = JobRunner(monitor)

# Storage for results and active jobs
job_results = {}
active_jobs = set()

In [4]:
# Add HTMX SSE extension after HTMX script
def get_htmx_idx(hdrs):
    return next((i for i, hdr in enumerate(hdrs) if (hdr.attrs.get('src') or '').endswith('htmx.min.js')), -1)

htmx_idx = get_htmx_idx(app.hdrs)
if htmx_idx >= 0:
    # Insert SSE extension right after HTMX
    app.hdrs.insert(htmx_idx+1, Script(src="https://unpkg.com/htmx-ext-sse"))
else:
    print("HTMX not found")

In [5]:
# Complex multi-stage pipeline
def data_pipeline(dataset_size=1000):
    from tqdm import tqdm
    import time
    
    results = {
        "stages": [],
        "metrics": {}
    }
    
    # Stage 1: Data Collection
    print("Starting Stage 1: Data Collection")
    collected_data = []
    for i in tqdm(range(dataset_size // 4), desc="1. Collecting from Source A"):
        time.sleep(0.002)
        collected_data.append(f"source_a_{i}")
    
    for i in tqdm(range(dataset_size // 4), desc="1. Collecting from Source B"):
        time.sleep(0.002)
        collected_data.append(f"source_b_{i}")
    
    results["stages"].append("Data Collection Complete")
    results["metrics"]["collected"] = len(collected_data)
    
    # Stage 2: Data Validation
    print("Starting Stage 2: Data Validation")
    valid_data = []
    for item in tqdm(collected_data, desc="2. Validating Data"):
        time.sleep(0.001)
        if random.random() > 0.1:  # 90% pass validation
            valid_data.append(item)
    
    results["stages"].append("Data Validation Complete")
    results["metrics"]["validated"] = len(valid_data)
    
    # Stage 3: Data Transformation
    print("Starting Stage 3: Data Transformation")
    
    # Sub-stage 3.1: Normalization
    normalized = []
    for item in tqdm(valid_data[:len(valid_data)//2], desc="3.1 Normalizing"):
        time.sleep(0.002)
        normalized.append(f"norm_{item}")
    
    # Sub-stage 3.2: Encoding
    encoded = []
    for item in tqdm(valid_data[len(valid_data)//2:], desc="3.2 Encoding"):
        time.sleep(0.002)
        encoded.append(f"enc_{item}")
    
    # Sub-stage 3.3: Merging
    merged = []
    for i in tqdm(range(min(len(normalized), len(encoded))), desc="3.3 Merging"):
        time.sleep(0.001)
        merged.append((normalized[i], encoded[i]))
    
    results["stages"].append("Data Transformation Complete")
    results["metrics"]["transformed"] = len(merged)
    
    # Stage 4: Analysis
    print("Starting Stage 4: Analysis")
    
    # Parallel analysis tasks
    for i in tqdm(range(100), desc="4.1 Statistical Analysis"):
        time.sleep(0.01)
    
    for i in tqdm(range(80), desc="4.2 Pattern Detection"):
        time.sleep(0.012)
    
    for i in tqdm(range(60), desc="4.3 Anomaly Detection"):
        time.sleep(0.015)
    
    results["stages"].append("Analysis Complete")
    results["metrics"]["patterns_found"] = random.randint(10, 50)
    results["metrics"]["anomalies"] = random.randint(1, 10)
    
    # Stage 5: Report Generation
    print("Starting Stage 5: Report Generation")
    
    for i in tqdm(range(50), desc="5.1 Generating Charts"):
        time.sleep(0.02)
    
    for i in tqdm(range(30), desc="5.2 Writing Summary"):
        time.sleep(0.03)
    
    for i in tqdm(range(20), desc="5.3 Finalizing Report"):
        time.sleep(0.04)
    
    results["stages"].append("Report Generation Complete")
    results["report"] = "pipeline_report.pdf"
    
    return results

In [6]:
# Nested loop example
def nested_processing():
    from tqdm import tqdm
    import time
    
    results = []
    
    # Outer loop - batches
    batches = 5
    items_per_batch = 20
    
    for batch in tqdm(range(batches), desc="Processing Batches"):
        batch_results = []
        
        # Inner loop - items in batch
        for item in tqdm(range(items_per_batch), desc=f"Batch {batch+1} Items", leave=False):
            time.sleep(0.05)
            batch_results.append(f"batch_{batch}_item_{item}")
        
        results.append(batch_results)
        time.sleep(0.1)  # Pause between batches
    
    return results

In [7]:
# Helper functions for UI rendering
def render_task_buttons(disabled=False, oob_swap=False):
    """Render task buttons with appropriate state"""
    btn_classes_pipeline = combine_classes(
        btn, 
        btn_colors.primary,
        btn_behaviors.disabled if disabled else ""
    )
    btn_classes_nested = combine_classes(
        btn, 
        btn_colors.secondary,
        btn_behaviors.disabled if disabled else ""
    )
    
    kwargs = {
        'id': 'task-buttons',
    }
    if oob_swap:
        kwargs['hx_swap_oob'] = 'true'
    
    return Div(
        Form(
            Input(type="hidden", name="task_type", value="pipeline"),
            Input(type="hidden", name="dataset_size", id="dataset-size-hidden", value="1000"),
            Button(
                "Start Pipeline", 
                type="submit",
                id="start-pipeline-btn",
                cls=btn_classes_pipeline,
                disabled=disabled,
                onclick="document.getElementById('dataset-size-hidden').value = document.getElementById('dataset-size').value" if not disabled else None
            ),
            hx_post="/start_task" if not disabled else None,
            hx_target="#progress-container",
            hx_swap="innerHTML",
            cls=combine_classes(display_tw.inline_block, m.r(0.5))
        ),
        Form(
            Input(type="hidden", name="task_type", value="nested"),
            Button(
                "Run Nested Task",
                type="submit",
                id="run-nested-btn",
                cls=btn_classes_nested,
                disabled=disabled
            ),
            hx_post="/start_task" if not disabled else None,
            hx_target="#progress-container",
            hx_swap="innerHTML",
            cls=combine_classes(display_tw.inline_block)
        ),
        **kwargs
    )

In [8]:
def render_stage_indicator(stage, status='pending'):
    """Render a single stage indicator badge"""
    if status == 'complete':
        badge_class = combine_classes(badge, badge_sizes.lg, badge_colors.success)
    elif status == 'active':
        badge_class = combine_classes(badge, badge_sizes.lg, badge_colors.warning)
    else:
        badge_class = combine_classes(badge, badge_sizes.lg, badge_styles.outline)
    
    return Span(stage, cls=badge_class, id=f"stage-{stage.lower().replace(' ', '-')}")

In [9]:
def detect_active_stages(bars):
    """Detect active and completed stages from progress bars"""
    stages = {
        'Collection': 'pending',
        'Validation': 'pending', 
        'Transformation': 'pending',
        'Analysis': 'pending',
        'Report': 'pending'
    }
    
    # Track which stages we've seen
    seen_stages = set()
    
    for bar_id, bar in bars.items():
        desc = bar.description or ""
        progress = bar.progress
        
        if '1.' in desc or 'Collecting' in desc:
            seen_stages.add('Collection')
            stages['Collection'] = 'complete' if progress >= 100 else 'active'
        elif '2.' in desc or 'Validating' in desc:
            seen_stages.add('Validation')
            stages['Validation'] = 'complete' if progress >= 100 else 'active'
            if 'Collection' not in seen_stages:
                stages['Collection'] = 'complete'
        elif '3.' in desc or 'Transform' in desc or 'Normaliz' in desc or 'Encod' in desc or 'Merg' in desc:
            seen_stages.add('Transformation')
            stages['Transformation'] = 'complete' if progress >= 100 else 'active'
            if 'Validation' not in seen_stages:
                stages['Validation'] = 'complete'
            if 'Collection' not in seen_stages:
                stages['Collection'] = 'complete'
        elif '4.' in desc or 'Analysis' in desc or 'Pattern' in desc or 'Anomaly' in desc:
            seen_stages.add('Analysis')
            stages['Analysis'] = 'complete' if progress >= 100 else 'active'
            for prev_stage in ['Collection', 'Validation', 'Transformation']:
                if prev_stage not in seen_stages:
                    stages[prev_stage] = 'complete'
        elif '5.' in desc or 'Report' in desc or 'Chart' in desc or 'Summary' in desc:
            seen_stages.add('Report')
            stages['Report'] = 'complete' if progress >= 100 else 'active'
            for prev_stage in ['Collection', 'Validation', 'Transformation', 'Analysis']:
                if prev_stage not in seen_stages:
                    stages[prev_stage] = 'complete'
    
    return stages

In [10]:
def get_bar_color(description):
    """Get progress bar color based on description"""
    if not description:
        return progress_colors.accent
    
    if '1.' in description:
        return progress_colors.info
    elif '2.' in description:
        return progress_colors.success
    elif '3.' in description:
        return progress_colors.warning
    elif '4.' in description:
        return progress_colors.error
    elif '5.' in description:
        return progress_colors.primary
    else:
        return progress_colors.accent

In [11]:
# Main page
@rt("/")
def index():
    return create_test_page(
        "Multi-Stage Progress Demo - SSE",
        Div(
            H1("Multi-Stage Pipeline Progress (SSE)", cls=combine_classes(font_size._3xl, font_weight.bold, m.b(6))),
            
            # Pipeline controls
            Div(
                H2("Pipeline Configuration", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Div(
                    Label("Dataset Size:", cls=str(label)),
                    Input(
                        id="dataset-size",
                        name="dataset_size",
                        type="range",
                        min="100",
                        max="2000",
                        value="1000",
                        cls=combine_classes(range_dui, range_colors.primary),
                        oninput="document.getElementById('size-display').textContent = this.value"
                    ),
                    Span("1000", id="size-display", cls=combine_classes(badge, badge_sizes.lg, m.l(2))),
                    cls=combine_classes(m.b(4))
                ),
                render_task_buttons(disabled=False),
                cls=combine_classes(card, bg_dui.base_200, p(6), m.b(6))
            ),
            
            # Progress container
            Div(id="progress-container", cls=str(m.b(6))),
            
            # Results
            Div(
                H2("Results", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Pre("No results yet", id="results", cls=combine_classes(bg_dui.base_300, p(4), rounded())),
                cls=combine_classes(card, bg_dui.base_200, p(6))
            ),
            
            cls=combine_classes(container, m.x.auto, p(8))
        )
    )

In [12]:
@rt("/start_task")
async def start_task(request):
    form = await request.form()
    task_type = form.get('task_type', 'pipeline')
    dataset_size = int(form.get('dataset_size', 1000))
    
    job_id = str(uuid.uuid4())
    active_jobs.add(job_id)
    
    if task_type == 'nested':
        task_name = "Nested Processing"
        def wrapper():
            result = nested_processing()
            job_results[job_id] = {"nested_results": result, "total_items": sum(len(b) for b in result)}
            active_jobs.discard(job_id)
            # Check monitor state after completion
            snapshot = monitor.snapshot(job_id)
            
        runner.start(
            job_id,
            wrapper,
            patch_kwargs={
                "min_delta_pct": 5,
                "min_update_interval": 0.1,
                "emit_initial": True
            }
        )
    else:
        task_name = "Data Pipeline"
        def wrapper():
            result = data_pipeline(dataset_size)
            job_results[job_id] = result
            active_jobs.discard(job_id)
            # Check monitor state after completion
            snapshot = monitor.snapshot(job_id)
    
        runner.start(
            job_id,
            wrapper,
            patch_kwargs={
                "min_delta_pct": 1,
                "min_update_interval": 0.05,
                "emit_initial": True
            }
        )
    
    # Stage indicators for pipeline tasks
    stages = ['Collection', 'Validation', 'Transformation', 'Analysis', 'Report'] if task_type == 'pipeline' else []
    
    # Return initial progress UI with SSE connections
    return Div(
        # Disable buttons via OOB swap
        render_task_buttons(disabled=True, oob_swap=True),
        # Progress display with SSE
        Div(
            H2("Pipeline Progress", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
            # Stage indicators (if pipeline)
            Div(
                *[render_stage_indicator(stage, 'pending') for stage in stages],
                cls=combine_classes(flex_display, flex_wrap.wrap, gap(2), m.b(4)),
                id="stage-indicators",
                hx_ext="sse" if stages else None,
                sse_connect=f"/stream_stages?job_id={job_id}" if stages else None,
                sse_swap="message" if stages else None
            ) if stages else None,
            # Overall progress with SSE
            Div(
                P("Overall Progress:", cls=str(font_weight.bold)),
                Progress(
                    value="0", 
                    max="100", 
                    cls=combine_classes(progress, progress_colors.primary, w.full, h(6)),
                    id="overall-progress"
                ),
                P("0%", cls=combine_classes(text_align.center, m.t(1)), id="overall-percent"),
                id="overall-progress-wrapper",
                hx_ext="sse",
                sse_connect=f"/stream_overall_progress?job_id={job_id}",
                sse_swap="message"
            ),
            # Active tasks container with SSE
            Div(
                H3("Starting...", cls=combine_classes(font_weight.semibold, m.b(3), m.t(6))),
                Div(
                    id="active-tasks",
                    cls=str(space.y(3))
                ),
                id="tasks-container",
                hx_ext="sse",
                sse_connect=f"/stream_tasks?job_id={job_id}",
                sse_swap="message"
            ),
            cls=combine_classes(card, bg_dui.base_200, p(6))
        )
    )

In [13]:
# SSE endpoint for overall progress
@rt("/stream_overall_progress")
def stream_overall_progress(job_id: str):
    """SSE endpoint for streaming overall progress updates"""
    
    async def progress_stream():
        update_count = 0
        try:
            async for data in sse_stream_async(
                monitor,
                job_id,
                interval=0.1,
                heartbeat=30.0,
                wait_for_start=True,
                start_timeout=5.0
            ):
                if data.startswith('data: '):
                    try:
                        json_str = data[6:].strip()
                        if json_str and json_str != '{}':
                            progress_data = json.loads(json_str)
                            update_count += 1
                            
                            if progress_data.get('completed'):                                
                                # Job completed - send final updates with OOB swaps
                                results = job_results.get(job_id, {})
                                
                                # Check if any other jobs are running
                                has_other_active = len(active_jobs) > 0
                                                                
                                yield sse_message(
                                    Div(
                                        # Final progress bar
                                        Div(
                                            P("Overall Progress:", cls=str(font_weight.bold)),
                                            Progress(
                                                value="100", 
                                                max="100", 
                                                cls=combine_classes(progress, progress_colors.primary, w.full, h(6)),
                                                id="overall-progress"
                                            ),
                                            P("100%", cls=combine_classes(text_align.center, m.t(1)), id="overall-percent"),
                                            id="overall-progress-wrapper",
                                            hx_swap_oob="true"
                                        ),
                                        # Re-enable buttons if no other active jobs
                                        render_task_buttons(disabled=has_other_active, oob_swap=True) if not has_other_active else "",
                                        # Update results
                                        Pre(
                                            json.dumps(results, indent=2),
                                            id="results",
                                            cls=combine_classes(bg_dui.base_300, p(4), rounded()),
                                            hx_swap_oob="true"
                                        )
                                    )
                                )
                                break
                            else:
                                # Update progress
                                progress_value = progress_data.get('progress', 0)
                                yield sse_message(
                                    Div(
                                        P("Overall Progress:", cls=str(font_weight.bold)),
                                        Progress(
                                            value=str(progress_value), 
                                            max="100", 
                                            cls=combine_classes(progress, progress_colors.primary, w.full, h(6)),
                                            id="overall-progress"
                                        ),
                                        P(f"{progress_value:.1f}%", cls=combine_classes(text_align.center, m.t(1)), id="overall-percent")
                                    )
                                )
                    except json.JSONDecodeError as e:
                        print(f"[DEBUG stream_overall_progress] JSON decode error: {e}, data: {json_str[:100]}")
                        pass
                elif data.startswith(': '):
                    yield data  # Heartbeat
        except Exception as e:
            print(f"[DEBUG stream_overall_progress] ERROR in overall progress stream for job {job_id[:8]}: {e}")
            import traceback
            traceback.print_exc()
    
    return EventStream(progress_stream())

In [14]:
# SSE endpoint for active tasks
@rt("/stream_tasks")
def stream_tasks(job_id: str):
    """SSE endpoint for streaming active task progress bars"""
    
    async def tasks_stream():
        update_count = 0
        last_progress_values = {}  # Track last progress values for each bar
        try:
            last_bars = {}
            final_bars = []  # Store final state of bars
            
            async for data in sse_stream_async(
                monitor,
                job_id,
                interval=0.1,
                heartbeat=30.0,
                wait_for_start=True,
                start_timeout=5.0
            ):
                if data.startswith('data: '):
                    try:
                        json_str = data[6:].strip()
                        if json_str and json_str != '{}':
                            progress_data = json.loads(json_str)
                            update_count += 1
                            
                            if progress_data.get('completed'):                                
                                # IMPORTANT: Fetch fresh snapshot to get the absolute final state
                                # This ensures we capture any last-millisecond updates
                                await asyncio.sleep(0.1)  # Small delay to ensure final updates are processed
                                final_snapshot = monitor.snapshot(job_id)
                                
                                if final_snapshot and final_snapshot['bars']:
                                    final_bars = []  # Rebuild with fresh data
                                    for bar_id, bar in final_snapshot['bars'].items():
                                        last_progress_values[bar_id] = bar.progress
                                        bar_color = get_bar_color(bar.description)
                                        bar_elem = Div(
                                            P(f"{bar.description}: {bar.progress:.1f}% ({bar.current}/{bar.total})", 
                                              cls=combine_classes(font_size.sm, font_weight.semibold, m.b(1))),
                                            Progress(
                                                value=str(bar.progress), 
                                                max="100", 
                                                cls=combine_classes(progress, bar_color, w.full)
                                            ),
                                            cls=combine_classes(p(3), bg_dui.base_300, rounded()),
                                            id=f"task-{bar_id}"
                                        )
                                        final_bars.append(bar_elem)
                                
                                # Keep the final state of progress bars and add completion message
                                yield sse_message(
                                    Div(
                                        H3("Active Tasks:", cls=combine_classes(font_weight.semibold, m.b(3), m.t(6))),
                                        Div(*final_bars, id="active-tasks", cls=str(space.y(3))),
                                        H3("Completed!", cls=combine_classes(font_weight.semibold, m.t(6), text_dui.success)),
                                        id="tasks-container",
                                        hx_swap_oob="true"
                                    )
                                )
                                break
                            
                            # Get current bars from snapshot
                            snapshot = monitor.snapshot(job_id)
                            if snapshot and snapshot['bars']:
                                bars_html = []
                                current_bar_ids = set()
                                
                                for bar_id, bar in snapshot['bars'].items():
                                    current_bar_ids.add(bar_id)
                                    last_progress_values[bar_id] = bar.progress
                                    
                                    bar_color = get_bar_color(bar.description)
                                    bar_elem = Div(
                                        P(f"{bar.description}: {bar.progress:.1f}% ({bar.current}/{bar.total})", 
                                          cls=combine_classes(font_size.sm, font_weight.semibold, m.b(1))),
                                        Progress(
                                            value=str(bar.progress), 
                                            max="100", 
                                            cls=combine_classes(progress, bar_color, w.full)
                                        ),
                                        cls=combine_classes(p(3), bg_dui.base_300, rounded()),
                                        id=f"task-{bar_id}"
                                    )
                                    bars_html.append(bar_elem)
                                    # Update final_bars to keep the last state
                                    final_bars = bars_html.copy()
                                
                                # Check for bar changes
                                if current_bar_ids != set(last_bars.keys()):
                                    added = current_bar_ids - set(last_bars.keys())
                                    removed = set(last_bars.keys()) - current_bar_ids
                                    last_bars = {bid: True for bid in current_bar_ids}
                                
                                # Always include the H3 heading with the tasks
                                yield sse_message(
                                    Div(
                                        H3("Active Tasks:", cls=combine_classes(font_weight.semibold, m.b(3), m.t(6))),
                                        Div(*bars_html, id="active-tasks", cls=str(space.y(3)))
                                    )
                                )
                    except json.JSONDecodeError as e:
                        print(f"[DEBUG stream_tasks] JSON decode error: {e}")
                        pass
                elif data.startswith(': '):
                    yield data  # Heartbeat
        except Exception as e:
            print(f"[DEBUG stream_tasks] ERROR in tasks stream for job {job_id[:8]}: {e}")
            import traceback
            traceback.print_exc()
    
    return EventStream(tasks_stream())

In [15]:
# SSE endpoint for stage indicators
@rt("/stream_stages")
def stream_stages(job_id: str):
    """SSE endpoint for streaming stage indicator updates"""
    
    async def stages_stream():
        update_count = 0
        try:
            last_stages = {}
            
            async for data in sse_stream_async(
                monitor,
                job_id,
                interval=0.1,
                heartbeat=30.0,
                wait_for_start=True,
                start_timeout=5.0
            ):
                if data.startswith('data: '):
                    try:
                        json_str = data[6:].strip()
                        if json_str and json_str != '{}':
                            progress_data = json.loads(json_str)
                            update_count += 1
                            
                            if progress_data.get('completed'):
                                # Mark all stages as complete
                                stages = ['Collection', 'Validation', 'Transformation', 'Analysis', 'Report']
                                yield sse_message(
                                    Div(
                                        *[render_stage_indicator(stage, 'complete') for stage in stages],
                                        cls=combine_classes(flex_display, flex_wrap.wrap, gap(2), m.b(4)),
                                        id="stage-indicators",
                                        hx_swap_oob="true"
                                    )
                                )
                                break
                            
                            # Get current stage statuses
                            snapshot = monitor.snapshot(job_id)
                            if snapshot and snapshot['bars']:
                                current_stages = detect_active_stages(snapshot['bars'])
                                
                                # Check if stages have changed
                                if current_stages != last_stages:
                                    changes = []
                                    for stage, status in current_stages.items():
                                        if last_stages.get(stage) != status:
                                            changes.append(f"{stage}: {last_stages.get(stage, 'unknown')} -> {status}")
                                                                        
                                    last_stages = current_stages.copy()
                                    
                                    # Send updated stage indicators
                                    stage_elements = []
                                    for stage, status in current_stages.items():
                                        stage_elements.append(render_stage_indicator(stage, status))
                                    
                                    yield sse_message(
                                        Div(
                                            *stage_elements,
                                            cls=combine_classes(flex_display, flex_wrap.wrap, gap(2), m.b(4))
                                        )
                                    )
                    except json.JSONDecodeError as e:
                        print(f"[DEBUG stream_stages] JSON decode error: {e}")
                        pass
                elif data.startswith(': '):
                    yield data  # Heartbeat
        except Exception as e:
            print(f"[DEBUG stream_stages] ERROR in stages stream for job {job_id[:8]}: {e}")
            import traceback
            traceback.print_exc()
    
    return EventStream(stages_stream())

In [18]:
# Start server for Jupyter
server = start_test_server(app)
display(HTMX())

In [19]:
# Stop server when done
server.stop()